Step 1:

KiteConnect API Connection!

URL: https://kite.zerodha.com/connect/login?v=3&api_key=wthlsr41oawqdeoz

Step 2:

Install the necessary libraries

In [ ]:
!pip install kiteconnect

In [ ]:
!pip install pandas-ta tensorflow keras-transformer

Step 3:

Paste the request_token copied from the URL to this input to obtain access_token

In [ ]:
from kiteconnect import KiteConnect
api_key="wthlsr41oawqdeoz"
api_secret="agvml7dbwicte27ijiswgpdqnlww2ho0"
kite = KiteConnect(api_key)
request_token = input("Request Token: ")
data = kite.generate_session(request_token, api_secret)
kite.set_access_token(data["access_token"])
access_token=data["access_token"]
access_token

Step 4:

Imports for the code and logic.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from kiteconnect import KiteConnect
import pandas_ta as ta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from time import sleep

# Your Kite object is already set up from previous cells
# We'll use 'kite' directly

print("All imports done")

In [ ]:
# Current nearest monthly future (as of Apr 25, 2026)
fut_symbol = "NFO:NIFTY26MAYFUT"  # Update if needed to next expiry

quote = kite.quote(fut_symbol)
inst_token = quote[fut_symbol]['instrument_token']

print("Instrument token:", inst_token)

Instrument token: 16914178


Nifty Future Spot Price

In [ ]:
spot_symbol = "NSE:NIFTY 50"  # Note the space
quote = kite.quote(spot_symbol)
spot_inst_token = quote[spot_symbol]['instrument_token']

print("Spot token:", spot_inst_token)
print("Current Nifty spot LTP:", quote[spot_symbol]['last_price'])

Spot token: 256265
Current Nifty spot LTP: 23897.95


In [ ]:
def fetch_data(from_date, to_date, interval='15minute'):
    data = kite.historical_data(spot_inst_token, from_date, to_date, interval)
    df = pd.DataFrame(data)
    if not df.empty:
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
    return df

print("Fetch function updated to 15-minute candles")

Fetch function updated to 15-minute candles


In [ ]:
# Training data: last 60 days up to yesterday
end_date = datetime(2026, 4, 24)  # Avoid partial today
start_date = end_date - timedelta(days=180)

df_train = fetch_data(start_date.strftime("%Y-%m-%d"),
                      end_date.strftime("%Y-%m-%d"))

print("Training data shape:", df_train.shape)
df_train.tail(2)

Training data shape: (3050, 5)


,open,high,low,close,volume
date,,,,,
2026-04-24 15:00:00+05:30,23920.65,23920.65,23863.3,23895.45,0
2026-04-24 15:15:00+05:30,23895.25,23924.20,23894.5,23903.95,0


In [ ]:
def add_indicators(df):
    df['EMA9'] = ta.ema(df['close'], length=9)
    df['EMA21'] = ta.ema(df['close'], length=21)
    df['RSI14'] = ta.rsi(df['close'], length=14)
    df['RSI6'] = ta.rsi(df['close'], length=6)   # Fast RSI for entry timing

    macd = ta.macd(df['close'])
    df['MACD'] = macd['MACD_12_26_9']
    df['MACD_signal'] = macd['MACDs_12_26_9']

    # Bollinger Bands (20,2) - standard
    bb = ta.bbands(df['close'], length=20, std=2)

    # --- Robust way to get BB column names ---
    bb_upper_col = next((col for col in bb.columns if 'BBU' in col), None)
    bb_middle_col = next((col for col in bb.columns if 'BBM' in col), None)
    bb_lower_col = next((col for col in bb.columns if 'BBL' in col), None)

    if bb_upper_col and bb_middle_col and bb_lower_col:
        df['BB_upper'] = bb[bb_upper_col]
        df['BB_middle'] = bb[bb_middle_col]
        df['BB_lower'] = bb[bb_lower_col]
    else:
        print("Could not find all Bollinger Band columns. Please check `bb.columns`.")
        print("bb.columns:", bb.columns.tolist())
        # Fallback or raise an error to indicate problem
        raise KeyError("Failed to find Bollinger Band columns (BBU, BBM, BBL)")
    # ----------------------------------------

    df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / df['BB_middle']  # Squeeze indicator
    df['BB_pct'] = (df['close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])  # %B

    # SuperTrend - use main trend column
    st = ta.supertrend(df['high'], df['low'], df['close'], length=10, multiplier=3)
    if st is not None and not st.empty:
        super_col = st.columns[0]
        df['SuperTrend'] = st[super_col]
    else:
        print("SuperTrend failed - skipping")
        df['SuperTrend'] = df['close']  # Fallback

    df.dropna(inplace=True)
    return df

df_train = add_indicators(df_train)

print("Indicators added. New shape:", df_train.shape)
df_train[['close', 'EMA9', 'EMA21', 'RSI14', 'RSI6', 'BB_upper', 'BB_lower', 'BB_pct', 'SuperTrend']].tail(3)

Indicators added. New shape: (3017, 17)


,close,EMA9,EMA21,RSI14,RSI6,BB_upper,BB_lower,BB_pct,SuperTrend
date,,,,,,,,,
2026-04-24 14:45:00+05:30,23920.90,23892.781111,23933.456674,43.921924,60.254950,23984.608268,23823.141732,0.605440,23955.250023
2026-04-24 15:00:00+05:30,23895.45,23893.314889,23930.001522,40.836656,49.870461,23968.043402,23829.966598,0.474253,23955.250023
2026-04-24 15:15:00+05:30,23903.95,23895.441911,23927.633202,42.294610,53.109315,23959.687469,23832.947531,0.560222,23955.250023


In [ ]:
# Update target to regression: predicted points change (next_close - current_close)
df_train['target_points'] = df_train['close'].shift(-1) - df_train['close']

# Drop the last row
df_train = df_train[:-1]

# Update y to regression target
y = df_train['target_points'].values

print("Regression target added. Shape:", y.shape)
print("Mean change:", np.mean(y), "Std:", np.std(y))

Regression target added. Shape: (3016,)
Mean change: -0.6552553050397878 Std: 48.7101080379365


In [ ]:
# Update target to regression: predicted points change (next_close - current_close)
df_train['target_points'] = df_train['close'].shift(-1) - df_train['close']

# Drop the last row
df_train = df_train[:-1]

# Update y to regression target
y = df_train['target_points'].values

print("Regression target added. Shape:", y.shape)
print("Mean change:", np.mean(y), "Std:", np.std(y))

In [ ]:
# Re-select features from updated df_train (12 features now including BB + RSI6)
features = ['close', 'EMA9', 'EMA21', 'RSI14', 'RSI6',
            'MACD', 'MACD_signal', 'SuperTrend',
            'BB_upper', 'BB_lower', 'BB_width', 'BB_pct']

X = df_train[features].values
y = df_train['target_points'].values  # points change

# Re-scale
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print("Fixed! X_scaled shape:", X_scaled.shape)
print("y (points) shape:", y.shape)

In [ ]:
timesteps = 60

X_seq = []
y_seq = []

for i in range(timesteps, len(X_scaled)):
    X_seq.append(X_scaled[i-timesteps:i])
    y_seq.append(y[i])  # predicted points change

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print("Regression sequences ready")
print("X_seq shape:", X_seq.shape)  # Expect (3009, 60, 7)
print("y_seq shape:", y_seq.shape)
print("Sample predicted moves:", y_seq[-5:])

In [ ]:
split = int(0.8 * len(X_seq))

X_train = X_seq[:split]
y_train = y_seq[:split]
X_val   = X_seq[split:]
y_val   = y_seq[split:]

print("Train samples:", X_train.shape[0])
print("Val samples:  ", X_val.shape[0])
print("Sample y_train moves:", y_train[-3:])
print("Sample y_val moves:  ", y_val[:3])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(timesteps, 12)))  # 12 features now
model.add(Dropout(0.3))

model.add(LSTM(64, return_sequences=False))
model.add(Dropout(0.3))

model.add(Dense(32, activation='relu'))
model.add(Dense(1))  # Linear output for points prediction

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='mse',
              metrics=['mae'])

model.summary()

In [ ]:
history = model.fit(X_train, y_train,
                    epochs=100,                  # More epochs for regression convergence
                    batch_size=32,
                    validation_data=(X_val, y_val),
                    verbose=1)

print("Regression training complete!")

In [ ]:
# Align df_val properly from df_train
val_start_idx = len(df_train) - len(X_val)  # Last rows for val
df_val = df_train.iloc[-len(X_val):].copy()

# ── BB + RSI Confluence Signals ──────────────────────────────────────────────
# Breakout: price crosses above upper band, RSI has momentum but not yet overbought
bb_bull_breakout = (df_val['close'] > df_val['BB_upper']) & (df_val['RSI14'].between(55, 75))
# Mean reversion: price touches lower band, RSI oversold, fast RSI turning up
bb_bull_reversion = (df_val['BB_pct'] < 0.05) & (df_val['RSI14'] < 35) & (df_val['RSI6'] > df_val['RSI6'].shift(1))

bb_bear_breakout  = (df_val['close'] < df_val['BB_lower']) & (df_val['RSI14'].between(25, 45))
bb_bear_reversion = (df_val['BB_pct'] > 0.95) & (df_val['RSI14'] > 65) & (df_val['RSI6'] < df_val['RSI6'].shift(1))

# Trend confirmation layer
trend_bull = (df_val['EMA9'] > df_val['EMA21']) & (df_val['close'] > df_val['SuperTrend'])
trend_bear = (df_val['EMA9'] < df_val['EMA21']) & (df_val['close'] < df_val['SuperTrend'])

df_val['bull_signal'] = (bb_bull_breakout | bb_bull_reversion) & trend_bull
df_val['bear_signal'] = (bb_bear_breakout | bb_bear_reversion) & trend_bear

# ── BB Squeeze Filter — suppress signals during low-volatility compression ───
bb_width_threshold = df_val['BB_width'].rolling(20).mean()
squeeze_active = df_val['BB_width'] < bb_width_threshold * 0.8  # 20% below avg width

df_val['bull_signal'] = df_val['bull_signal'] & ~squeeze_active
df_val['bear_signal'] = df_val['bear_signal'] & ~squeeze_active

trades = df_val['bull_signal'] | df_val['bear_signal']
direction = np.where(df_val['bull_signal'], 1, -1)  # 1 bull, -1 bear

print(f"Total signals: {trades.sum()}")
print(f"Percentage of bars: {trades.mean()*100:.1f}%")
print(f"Bull signals: {df_val['bull_signal'].sum()} | Bear signals: {df_val['bear_signal'].sum()}")
print(f"Bars suppressed by squeeze filter: {squeeze_active.sum()}")

# Next candle high/low for hit check
df_val['next_high'] = df_val['high'].shift(-1)
df_val['next_low'] = df_val['low'].shift(-1)

bull_trades = trades & (direction == 1)
bear_trades = trades & (direction == -1)

bull_wins = bull_trades & (df_val['next_high'] >= df_val['close'] + 5)
bear_wins = bear_trades & (df_val['next_low'] <= df_val['close'] - 5)

wins = bull_wins | bear_wins

win_rate = wins.sum() / trades.sum() if trades.sum() > 0 else 0
avg_bull_hit = (df_val['next_high'][bull_wins] - df_val['close'][bull_wins]).mean()
avg_bear_hit = (df_val['close'][bear_wins] - df_val['next_low'][bear_wins]).mean()

print(f"\nWin rate (hit >=5 pts in dir): {win_rate*100:.1f}%")
print(f"Winning trades: {wins.sum()} / {trades.sum()}")
print(f"Avg bull hit: {avg_bull_hit:.1f} points")
print(f"Avg bear hit: {avg_bear_hit:.1f} points")

Total signals: 26
Percentage of bars: 4.4%
Bull signals: 14 | Bear signals: 12
Bars suppressed by squeeze filter: 203

Win rate (hit >=5 pts in dir): 96.2%
Winning trades: 25 / 26
Avg bull hit: 40.4 points
Avg bear hit: 29.0 points


Step 5:

Run this cell for a few times until you get the signal as UP or DOWN.
Once the signal is confirmed, then only run the remaining code.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def get_current_indicators():
    # Fetch last 50 15-min candles of Nifty spot (enough for indicators)
    end = datetime.now()
    start = end - timedelta(days=7)  # safe buffer
    df = fetch_data(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"), '15minute')
    df = add_indicators(df)

    # Latest completed candle
    latest = df.iloc[-1]
    prev   = df.iloc[-2]

    current_price = latest['close']
    ema9       = latest['EMA9']
    ema21      = latest['EMA21']
    rsi        = latest['RSI14']
    rsi6       = latest['RSI6']
    prev_rsi6  = prev['RSI6']
    supertrend = latest['SuperTrend']
    bb_upper   = latest['BB_upper']
    bb_lower   = latest['BB_lower']
    bb_pct     = latest['BB_pct']
    bb_width   = latest['BB_width']

    # Squeeze detection: current width vs 20-bar rolling average
    avg_bb_width = df['BB_width'].rolling(20).mean().iloc[-1]
    in_squeeze   = bb_width < avg_bb_width * 0.8

    # ── BB + RSI Confluence Signals ────────────────────────────────────────
    bb_bull_breakout  = (current_price > bb_upper) and (55 <= rsi <= 75)
    bb_bull_reversion = (bb_pct < 0.05) and (rsi < 35) and (rsi6 > prev_rsi6)
    bb_bear_breakout  = (current_price < bb_lower) and (25 <= rsi <= 45)
    bb_bear_reversion = (bb_pct > 0.95) and (rsi > 65) and (rsi6 < prev_rsi6)

    trend_bull = (ema9 > ema21) and (current_price > supertrend)
    trend_bear = (ema9 < ema21) and (current_price < supertrend)

    bull = (bb_bull_breakout or bb_bull_reversion) and trend_bull and not in_squeeze
    bear = (bb_bear_breakout or bb_bear_reversion) and trend_bear and not in_squeeze

    direction = "UP" if bull else "DOWN" if bear else "NO_SIGNAL"

    print(f"Current Nifty Spot: {current_price:.2f}")
    print(f"EMA9: {ema9:.2f} | EMA21: {ema21:.2f} | RSI14: {rsi:.2f} | RSI6: {rsi6:.2f} | SuperTrend: {supertrend:.2f}")
    print(f"BB Upper: {bb_upper:.2f} | BB Lower: {bb_lower:.2f} | %B: {bb_pct:.2f} | Width: {bb_width:.4f}")
    print(f"Squeeze Active: {in_squeeze} | Avg BB Width: {avg_bb_width:.4f}")
    print(f"Signal: {direction}")

    return direction, current_price

# Test it
direction, spot_price = get_current_indicators()

Current Nifty Spot: 23903.95
EMA9: 23895.44 | EMA21: 23927.63 | RSI14: 42.28 | RSI6: 53.11 | SuperTrend: 23955.25
BB Upper: 23959.69 | BB Lower: 23832.95 | %B: 0.56 | Width: 0.0053
Squeeze Active: True | Avg BB Width: 0.0153
Signal: NO_SIGNAL


In [ ]:
def get_atm_option(direction, spot_price):
    expiry = "26505"  # Update to current weekly
    option_type = "CE" if direction == "UP" else "PE"

    base_strike = round(spot_price / 100) * 100  # Nearest 100
    lower_range = base_strike - 500
    upper_range = base_strike + 500
    strikes = list(range(lower_range, upper_range, 100))  #

    candidates = []
    for strike in strikes:
        symbol = f"NFO:NIFTY{expiry}{strike}{option_type}"
        try:
            quote = kite.quote(symbol)
            premium = quote[symbol]['last_price']
            if 80 <= premium <= 100:
                candidates.append((symbol, premium, strike))
        except:
            pass

    if candidates:
        # Pick closest to spot
        candidates.sort(key=lambda x: abs(x[2] - spot_price))
        best = candidates[0]
        print(f"Selected: {best[0]} @ {best[1]} (strike {best[2]})")
        return best[0]
    else:
        print("No option in 150-180 range")
        return None

# Test
option_symbol = get_atm_option(direction, spot_price)
print(option_symbol)

Selected: NFO:NIFTY2650523600PE @ 175.15 (strike 23600)
NFO:NIFTY2650523600PE


In [ ]:
max_daily_loss = -5000  # Your cap

def get_today_realised_pnl():
    try:
        trades = kite.trades()  # All executed trades today
        realised_pnl = 0
        for trade in trades:
            realised_pnl += trade.get('realised_profit', 0)  # Direct realised P&L per trade
        print(f"Today realised P&L: ₹{realised_pnl:.0f}")
        return realised_pnl
    except Exception as e:
        print("PNL fetch failed:", e)
        return 0

def check_daily_cap():
    today_pnl = get_today_realised_pnl()
    if today_pnl <= max_daily_loss:
        print(f"Daily loss cap hit: ₹{abs(today_pnl):.0f} loss. No more trades today.")
        return False
    print(f"Daily cap safe: ₹{today_pnl:.0f} P&L")
    return True

In [ ]:
check_daily_cap()

Today realised P&L: ₹0
Daily cap safe: ₹0 P&L


True

Time Logic Variables

In [ ]:
from datetime import datetime, timedelta
import pandas as pd
from time import sleep
import warnings
warnings.filterwarnings("ignore")

today_now = datetime.now()
today = today_now + timedelta(hours=9)
yesterday = today_now - timedelta(hours=72)
past_dated = today_now - timedelta(hours=1440)
training_dated = today_now - timedelta(hours=96)
today = datetime.strftime(today, "%Y-%m-%d %H:%M:%S")
yesterday = datetime.strftime(yesterday, "%Y-%m-%d %H:%M:%S")

Step 6:

Main Code!

Run this code for real trades.

In [ ]:
def Time():
    orders = kite.orders()
    orders = pd.DataFrame(orders)
    x = len(orders)-1
    order_time = orders['order_timestamp'][x]
    time = datetime.now() - order_time
    time = pd.Timedelta(time)
    td = round(time.total_seconds())
    return td

profit = []
loss = []

for i in range(5):
    direction, spot_price = get_current_indicators()
    print(f"Direction is {direction} at spot price {spot_price}")
    tsymbol = get_atm_option(direction, spot_price)
    x = len(kite.orders())
    buy_order = kite.place_order(
                                tradingsymbol=tsymbol,
                                    exchange=kite.EXCHANGE_NFO,
                                    transaction_type=kite.TRANSACTION_TYPE_BUY,
                                    quantity=500,
                                    variety=kite.VARIETY_REGULAR,
                                    order_type=kite.ORDER_TYPE_MARKET,
                                    product=kite.PRODUCT_MIS
    )
    orders = kite.orders()
    orders = pd.DataFrame(orders)
    avg_price = orders['average_price'][x]
    print("The Buy Order was executed at: {}".format(avg_price))
    x += 1
    sleep(0.5)

    target = round(avg_price * 1.05, 1)
    sell_order_id = kite.place_order(tradingsymbol=tsymbol,
                                    exchange=kite.EXCHANGE_NFO,
                                    transaction_type=kite.TRANSACTION_TYPE_SELL,
                                    quantity=500,
                                    price=target,
                                    variety=kite.VARIETY_REGULAR,
                                    order_type=kite.ORDER_TYPE_LIMIT,
                                    product=kite.PRODUCT_MIS)
    print("The SELL Order placed at a price of : {}".format(target))

    t = Time()
    while t < 60:
        all_orders = kite.orders()
        all_orders = pd.DataFrame(all_orders)
        last = len(all_orders) - 1
        order_status = all_orders['status'][last]
        if order_status == 'COMPLETE':
            target_avg = all_orders['average_price'][last]
            print("The Target got executed at: ", target_avg)
            profit.append(target_avg - avg_price)
            sleep(1)
            break
        sleep(1)
        t = Time()
        print("Waiting for target to get executed...")

    all_orders = kite.orders()
    all_orders = pd.DataFrame(all_orders)
    last = len(all_orders) - 1
    order_status = all_orders['status'][last]
    sl = all_orders['order_id'][last]

    if (order_status == 'OPEN'):
        stoploss = kite.modify_order(order_id=sl,
                                    quantity=500,
                                    variety=kite.VARIETY_REGULAR,
                                    order_type=kite.ORDER_TYPE_MARKET)
        all_orders = pd.DataFrame(kite.orders())
        last = len(all_orders) - 1
        sell_avg = all_orders['average_price'][last]
        print("The Stop Loss was executed at: ", sell_avg)
        sleep(1)
        loss.append(sell_avg - avg_price)

    print("Iteration ", (i+1), " has been completed!")